# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
spaceship.shape

(8693, 14)

In [4]:
spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

In [5]:
spaceship.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [6]:
spaceship.dropna(inplace=True)

In [7]:
spaceship["deck"] = spaceship["Cabin"].apply(lambda x: x[:1])

In [8]:
spaceship.drop(["PassengerId", "Name"], axis=1, inplace=True)

In [9]:
spaceship_categorical = spaceship.select_dtypes("object")
spaceship_categorical = pd.get_dummies(spaceship_categorical.drop("Cabin", axis=1), dtype=int)
spaceship_categorical

,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_False,CryoSleep_True,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_False,VIP_True,deck_A,deck_B,deck_C,deck_D,deck_E,deck_F,deck_G,deck_T
0,0,1,0,1,0,0,0,1,1,0,0,1,0,0,0,0,0,0
1,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,0,0
2,0,1,0,1,0,0,0,1,0,1,1,0,0,0,0,0,0,0
3,0,1,0,1,0,0,0,1,1,0,1,0,0,0,0,0,0,0
4,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,0,1,0,1,0,1,0,0,0,1,1,0,0,0,0,0,0,0
8689,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,1,0
8690,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,1,0
8691,0,1,0,1,0,1,0,0,1,0,0,0,0,0,1,0,0,0


In [10]:
features = pd.concat([spaceship.select_dtypes("number"), spaceship_categorical], axis=1)
features

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_False,...,VIP_False,VIP_True,deck_A,deck_B,deck_C,deck_D,deck_E,deck_F,deck_G,deck_T
0,39.0,0.0,0.0,0.0,0.0,0.0,0,1,0,1,...,1,0,0,1,0,0,0,0,0,0
1,24.0,109.0,9.0,25.0,549.0,44.0,1,0,0,1,...,1,0,0,0,0,0,0,1,0,0
2,58.0,43.0,3576.0,0.0,6715.0,49.0,0,1,0,1,...,0,1,1,0,0,0,0,0,0,0
3,33.0,0.0,1283.0,371.0,3329.0,193.0,0,1,0,1,...,1,0,1,0,0,0,0,0,0,0
4,16.0,303.0,70.0,151.0,565.0,2.0,1,0,0,1,...,1,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,41.0,0.0,6819.0,0.0,1643.0,74.0,0,1,0,1,...,0,1,1,0,0,0,0,0,0,0
8689,18.0,0.0,0.0,0.0,0.0,0.0,1,0,0,0,...,1,0,0,0,0,0,0,0,1,0
8690,26.0,0.0,0.0,1872.0,1.0,0.0,1,0,0,1,...,1,0,0,0,0,0,0,0,1,0
8691,32.0,0.0,1049.0,0.0,353.0,3235.0,0,1,0,1,...,1,0,0,0,0,0,1,0,0,0


In [11]:
target = spaceship["Transported"]

**Perform Train Test Split**

In [12]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.20)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [ ]:
# Bagging

In [13]:
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols)
], remainder='passthrough')

In [16]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

dt = DecisionTreeClassifier(random_state=42)

bagging = BaggingClassifier(
    estimator=dt,
    n_estimators=100,
    max_samples=0.8,
    bootstrap=True,
    random_state=42
)

scores_bagging = cross_val_score(
    bagging,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Bagging mean CV accuracy:", scores_bagging.mean())

Bagging mean CV accuracy: 0.7880355709412001


In [18]:
###
from sklearn.pipeline import Pipeline

pipeline_bagging = Pipeline([
    ('preprocessor', preprocessor),
    ('model', bagging)
])

scores_bagging = cross_val_score(
    pipeline_bagging,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Bagging mean CV accuracy:", scores_bagging.mean())
####

Bagging mean CV accuracy: 0.7882249648805941


In [ ]:
## Pasting

In [19]:
pasting = BaggingClassifier(
    estimator=dt,
    n_estimators=100,
    max_samples=0.8,
    bootstrap=False,  # ⭐ Pasting
    random_state=42
)

pipeline_pasting = Pipeline([
    ('preprocessor', preprocessor),
    ('model', pasting)
])

scores_pasting = cross_val_score(
    pipeline_pasting,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Pasting mean CV accuracy:", scores_pasting.mean())

Pasting mean CV accuracy: 0.7872810412545512


- Random Forests

In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

scores_rf = cross_val_score(
    rf,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Random Forest mean CV accuracy:", scores_rf.mean())
print("Std:", scores_rf.std())

Random Forest mean CV accuracy: 0.7870930807602993
Std: 0.005182486864600406


In [21]:
### Option 2/ pipeline (must have preprocessor)
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', rf)
])

scores_rf = cross_val_score(
    pipeline_rf,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Random Forest mean CV accuracy:", scores_rf.mean())

Random Forest mean CV accuracy: 0.7869038660015482


- Gradient Boosting

In [22]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

scores_gb = cross_val_score(
    gb,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Gradient Boosting mean CV accuracy:", scores_gb.mean())
print("Std:", scores_gb.std())

Gradient Boosting mean CV accuracy: 0.7997683194289154
Std: 0.013779209542006059


In [24]:
### Option 2
from sklearn.pipeline import Pipeline

pipeline_gb = Pipeline([
    ('preprocessor', preprocessor),  # nếu bạn dùng ở RF
    ('model', GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ))
])

scores_gb = cross_val_score(
    pipeline_gb,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("Gradient Boosting mean CV accuracy:", scores_gb.mean())

Gradient Boosting mean CV accuracy: 0.7995789254895215


- Adaptive Boosting

In [25]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

base_tree = DecisionTreeClassifier(
    max_depth=1,
    random_state=42
)

ada = AdaBoostClassifier(
    estimator=base_tree,
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

scores_ada = cross_val_score(
    ada,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print("AdaBoost mean CV accuracy:", scores_ada.mean())
print("Std:", scores_ada.std())

AdaBoost mean CV accuracy: 0.740345388606978
Std: 0.010205575773523589


Which model is the best and why?

Random Forest is the best-performing model because it achieved the highest accuracy and the most stable performance across cross-validation folds.

1. Highest accuracy

It achieved the best mean cross-validation accuracy, indicating strong predictive performance.

2. Stable performance

Low standard deviation across folds

This shows the model is robust and not sensitive to data splits.

3. Good bias–variance trade-off

By averaging many decision trees trained on different subsets of data and features, Random Forest:

Reduces overfitting

Improves generalization

4. Works well with mixed features

Handles numerical + one-hot encoded categorical features effectively

No strict requirement for feature scaling